<a href="https://colab.research.google.com/github/hanidew/G12_IR_Evaluation/blob/main/IRWS_G12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Data Cleaning (Hani)

Check 50 topics each topic got 1000 documents

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [12]:
import pandas as pd
import glob
import os
import csv

print("1. Loading System Runs...")
run_cols = ['Topic_ID', 'Iteration', 'Doc_ID', 'Rank', 'Score', 'Trash']

# UPDATE THIS PATH to your folder containing the 15 system runs
run_files_path = '/content/drive/MyDrive/IRWS_G12/set12/System_Runs/*'

all_runs_list = []
for file in glob.glob(run_files_path):
    # Parse the documents
    df = pd.read_csv(file, sep=r'\s+', names=run_cols, quoting=csv.QUOTE_NONE, na_filter=False)

    # Extract the true system name from the filename
    df['System_Name'] = os.path.basename(file).split('.')[-1]

    all_runs_list.append(df)

# Combine into one DataFrame
full_runs_df = pd.concat(all_runs_list, ignore_index=True)

# Drop the broken internal system name column and ensure Topic_ID is a string
full_runs_df.drop(columns=['Trash'], inplace=True)
full_runs_df['Topic_ID'] = full_runs_df['Topic_ID'].astype(str)

print("Data successfully loaded!")

print("\n2. Running Depth Audit...")
# Group by System and Topic to count documents
depth_audit = full_runs_df.groupby(['System_Name', 'Topic_ID'])['Doc_ID'].count().reset_index()
depth_audit.rename(columns={'Doc_ID': 'Document_Count'}, inplace=True)

# Filter for any anomalies
anomalies = depth_audit[depth_audit['Document_Count'] < 1000]

print(f"Total system/topic combinations checked: {len(depth_audit)}")
print(f"Total anomalies found (< 1000 docs): {len(anomalies)}")

if len(anomalies) > 0:
    print("\nAnomalies are still present in the new files. Here they are:")
    print(anomalies)
else:
    print("\nPERFECT RUN! All 15 systems successfully retrieved exactly 1000 documents per topic!")

1. Loading System Runs...
Data successfully loaded!

2. Running Depth Audit...
Total system/topic combinations checked: 750
Total anomalies found (< 1000 docs): 0

PERFECT RUN! All 15 systems successfully retrieved exactly 1000 documents per topic!


In [13]:
print("\n3. Standardizing Ranks (1-1000)...")

# Sort to guarantee highest scores are at the top
full_runs_df.sort_values(
    by=['System_Name', 'Topic_ID', 'Score'],
    ascending=[True, True, False],
    inplace=True
)

# Overwrite old messy ranks with a fresh 1 to 1000 sequence
full_runs_df['Rank'] = full_runs_df.groupby(['System_Name', 'Topic_ID']).cumcount() + 1

print("\n4. Running Final Verification...")

# Group by System and Topic to check our work
verification = full_runs_df.groupby(['System_Name', 'Topic_ID']).agg(
    Document_Count=('Doc_ID', 'count'),
    Min_Rank=('Rank', 'min'),
    Max_Rank=('Rank', 'max')
).reset_index()

# Find any groups that failed criteria
failed_groups = verification[
    (verification['Document_Count'] != 1000) |
    (verification['Min_Rank'] != 1) |
    (verification['Max_Rank'] != 1000)
]

if len(failed_groups) > 0:
    print("WARNING: Standardization failed for the following groups:")
    print(failed_groups)
else:
    print("SUCCESS: Verification Passed! All 750 groups have exactly 1000 documents ranked 1 to 1000.")



3. Standardizing Ranks (1-1000)...

4. Running Final Verification...
SUCCESS: Verification Passed! All 750 groups have exactly 1000 documents ranked 1 to 1000.


In [14]:

    # --- THE FINAL DATA HANDOFF ---
    export_path = '/content/drive/MyDrive/IRWS_G12/set12/clean_system_runs_FINAL.csv'
    full_runs_df.to_csv(export_path, index=False)

    print(f"\n--- DATA HANDOFF READY ---")
    print(f"Data has been saved to: {export_path}")


--- DATA HANDOFF READY ---
Data has been saved to: /content/drive/MyDrive/IRWS_G12/set12/clean_system_runs_FINAL.csv
